# CareTrace — Neurosymbolic Pediatric Triage

**UC Berkeley DATASCI 290 — Neurosymbolic AI (Spring 2026 Final Project)**

**Team:** Annas (Orchestrate) · Yoko (Extract) · David (Knowledge Graph) · Alex (Rules)

---

## What this notebook shows

CareTrace is a pediatric fever-triage agent that combines four stacked layers:

1. **LLM interpretation** — pulls structured clinical facts out of a caregiver's natural-language message
2. **Knowledge graph grounding** — maps surface symptoms to SNOMED CT concepts and red-flag rules (Neo4j, with a dict fallback)
3. **Symbolic rules** — a 3-layer plain-Python pipeline (observation → concern → decision) encoding Seattle Children's Fever CPG
4. **LLM explanation** — turns the structured decision into empathetic caregiver-facing guidance

The LLM never makes the clinical decision. It extracts facts on the way in and verbalizes the rules' decision on the way out. Everything safety-critical happens in symbolic code, so every disposition is fully traceable.

## Architecture

```
┌─────────────┐    ┌──────────────┐    ┌───────────────┐    ┌─────────────┐
│ Caregiver   │───▶│ Interpret    │───▶│ Normalize     │───▶│ Evaluate    │
│ message     │    │ (LLM → facts)│    │ (KG + flags)  │    │ (rules)     │
└─────────────┘    └──────────────┘    └───────────────┘    └──────┬──────┘
                                                                   │
                                        ┌──────────────────────────┴─┐
                                        │ disposition?                │
                                        ├─────────────────┬───────────┤
                                        ▼                 ▼
                                ┌───────────────┐  ┌────────────────┐
                                │ Explain       │  │ Ask follow-up  │
                                │ (LLM verbalize)│  │ (structured Q) │
                                └───────┬───────┘  └────────┬───────┘
                                        │                   │
                                        └────────┬──────────┘
                                                 ▼
                                         Caregiver reply

Dispositions (Alex's vocab):  er_now  |  urgent_eval  |  home_monitor  |  unsupported
```


## 0. Setup

Import the consolidated `caretrace` package and load the Groq API key. The KG adapter auto-detects Neo4j; if it's not reachable we fall back to an in-memory dict of SNOMED concepts.

In [1]:
import os, sys, json
from pprint import pprint
from pathlib import Path

# Make sure we can import the caretrace package from this notebook's parent
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

# Load .env so Groq + Neo4j credentials become available
from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

print("GROQ_API_KEY set:     ", bool(os.getenv("GROQ_API_KEY")))
print("NEO4J_URI set:        ", bool(os.getenv("NEO4J_URI")))
print("KG backend preference:", os.getenv("CARETRACE_KG_BACKEND", "auto"))

GROQ_API_KEY set:      True
NEO4J_URI set:         True
KG backend preference: auto


## 1. Shared state — `ClinicalState`

Every node in the LangGraph reads and writes a single `ClinicalState` TypedDict. The most important field is `facts` — a nested dict using Alex's categorical vocabulary that the rules engine consumes directly. Raw typed fields (`age_months`, `temperature_f`, `current_medication`) live at the top level because the KG and fallback red flags need them.


In [2]:
from caretrace.state import (
    ClinicalState,
    FACT_KEYS,
    REQUIRED_FACTS_FOR_HOME,
    FACT_QUESTIONS,
    DISPOSITION_SEVERITY,
    initial_state,
)

print("FACT_KEYS:              ", FACT_KEYS)
print("REQUIRED_FACTS_FOR_HOME:", REQUIRED_FACTS_FOR_HOME)
print()
print("Initial (blank) state keys:")
for k in initial_state().keys():
    print(f"  - {k}")

FACT_KEYS:               ['fever', 'alert', 'intake', 'urination', 'vomiting', 'breathing', 'seizure', 'rash']
REQUIRED_FACTS_FOR_HOME: ['alert', 'breathing', 'intake', 'urination']

Initial (blank) state keys:
  - messages
  - turn
  - facts
  - age_months
  - temperature_f
  - fever_duration_days
  - current_medication
  - medication_last_dose
  - weight_kg
  - raw_symptoms
  - kg_backend
  - grounded_concepts
  - all_sctids
  - kg_red_flags
  - observation_predicates
  - concern_predicates
  - decision
  - rules_triggered
  - disposition
  - missing_required
  - follow_up_question
  - explanation
  - key_positives
  - key_negatives
  - go_now_thresholds
  - overnight_plan
  - is_complete
  - phase


In [3]:
# Disposition severity ranking — 'merge worst wins' semantics
print("Disposition priority (lower = more severe):")
for disp, rank in sorted(DISPOSITION_SEVERITY.items(), key=lambda x: x[1]):
    print(f"  {rank}: {disp}")

Disposition priority (lower = more severe):
  0: er_now
  1: urgent_eval
  2: home_monitor
  3: unsupported
  4: None


## 2. Knowledge graph adapter

`KnowledgeAdapter` wraps David's `KnowledgeRetrievalAgent` and falls back to a dict of SNOMED concepts when Neo4j isn't reachable. Both backends expose the same `ground_symptoms(mentions) → {results, all_sctids}` and `get_red_flags(sctids) → [...]` shape, so the rest of the pipeline is backend-agnostic.


In [4]:
from caretrace.kg.adapter import KnowledgeAdapter

# prefer='auto' — tries Neo4j, falls back to dict
adapter = KnowledgeAdapter(prefer="auto")
print(f"Active backend: {adapter.backend}")

Active backend: dict


In [5]:
# Ground a set of mentions. The adapter walks IS_A ancestors and returns
# SCTIDs for every matched concept.
mentions = ["fever", "lethargy", "vomiting", "reduced fluid intake"]
grounding = adapter.ground_symptoms(mentions)

print("Grounded concepts:")
for r in grounding["results"]:
    print(f"  {r.get('mention')!r:30s} → {r.get('concepts', [])[:2]}")
print()
print(f"All SCTIDs: {grounding['all_sctids'][:10]}{'...' if len(grounding['all_sctids']) > 10 else ''}")
print(f"Ungrounded: {grounding.get('ungrounded', [])}")

Grounded concepts:
  'fever'                        → [{'sctid': '386661006', 'fsn': 'Fever', 'type': 'finding', 'depth': 0, 'ancestors': [{'fsn': 'Finding of body temperature', 'sctid': None}, {'fsn': 'Clinical finding', 'sctid': None}]}]
  'lethargy'                     → [{'sctid': '214264003', 'fsn': 'Lethargy', 'type': 'finding', 'depth': 0, 'ancestors': [{'fsn': 'Alteration of consciousness', 'sctid': None}, {'fsn': 'Neurological finding', 'sctid': None}]}]
  'vomiting'                     → [{'sctid': '422400008', 'fsn': 'Vomiting', 'type': 'finding', 'depth': 0, 'ancestors': [{'fsn': 'Gastrointestinal finding', 'sctid': None}, {'fsn': 'Clinical finding', 'sctid': None}]}]
  'reduced fluid intake'         → [{'sctid': '271795006', 'fsn': 'Reduced fluid intake', 'type': 'finding', 'depth': 0, 'ancestors': [{'fsn': 'Finding of fluid intake', 'sctid': None}, {'fsn': 'Clinical finding', 'sctid': None}]}]

All SCTIDs: ['422400008', '214264003', '271795006', '386661006']
Ungrounded: [

In [6]:
# Red flags from the KG, keyed on the grounded SCTIDs
kg_flags = adapter.get_red_flags(grounding["all_sctids"])
print(f"KG red flags for these concepts ({len(kg_flags)}):")
for f in kg_flags:
    print(f"  [{f.get('source', 'kg'):8s}] {f['rule_id']:20s} → {f['disposition']:12s}  {f.get('description', '')[:60]}")

KG red flags for these concepts (0):


## 3. Fallback red flags

When Neo4j is offline, Alex's rules alone miss high-severity signals like infant fever, seizure, and breathing difficulty (because those aren't in the dehydration-focused fever CPG). `check_fallback_red_flags` runs a small dict-based rule set that catches them and emits the same red-flag shape the safety layer expects.


In [7]:
from caretrace.kg.fallback_red_flags import check_fallback_red_flags, highest_disposition

# Build a few sample states showing each fallback rule firing
samples = [
    ("2mo with any fever",
        {"age_months": 2, "facts": {"fever": "yes"}}),
    ("5yo with seizure",
        {"age_months": 60, "facts": {"fever": "yes", "seizure": "yes"}}),
    ("4yo with breathing difficulty",
        {"age_months": 48, "facts": {"fever": "yes", "breathing": "difficulty"}}),
    ("Temperature >= 104°F",
        {"age_months": 60, "temperature_f": 104.2, "facts": {"fever": "yes"}}),
    ("Rash + fever",
        {"age_months": 60, "facts": {"fever": "yes", "rash": "yes"}}),
]

for label, state in samples:
    flags = check_fallback_red_flags(state)
    print(f"— {label}")
    for f in flags:
        print(f"    {f['rule_id']:20s} → {f['disposition']:12s}  ({f['description']})")
    print(f"    highest: {highest_disposition(flags)}")
    print()

— 2mo with any fever
    FB_INFANT_FEVER      → er_now        (Any fever in an infant under 3 months is an emergency.)
    highest: er_now

— 5yo with seizure
    FB_SEIZURE           → er_now        (A febrile seizure requires immediate emergency evaluation.)
    highest: er_now

— 4yo with breathing difficulty
    FB_BREATHING         → er_now        (Difficulty breathing in a febrile child is an emergency.)
    highest: er_now

— Temperature >= 104°F
    FB_VERY_HIGH_FEVER   → urgent_eval   (Temperature ≥104°F warrants urgent evaluation.)
    highest: urgent_eval

— Rash + fever
    FB_RASH_FEVER        → urgent_eval   (New rash with fever — check for non-blanching spots urgently.)
    highest: urgent_eval



## 4. Interpretation agent — LLM → structured facts

The interpretation agent uses Groq's Llama 3.3 70B with Pydantic `with_structured_output` to extract clinical facts from a single caregiver message. It outputs Yoko's binary vocabulary (`alert: yes/no`, `drinking: yes/some/no`), then `_translate_to_facts` maps those onto Alex's categorical vocabulary. The LLM is forbidden from giving any advice, diagnosis, or disposition — its only job is extraction.


In [8]:
from langchain_core.messages import HumanMessage
from caretrace.agents.interpretation import interpret, ExtractionResult, _translate_to_facts, _derive_raw_symptoms

# Build a fresh state with one caregiver message
state = initial_state()
state["messages"] = [HumanMessage(content=(
    "My 6-year-old has had a fever since yesterday. Temperature is 101.8. "
    "He's awake and talking to me, drinking water normally, breathing fine. "
    "He threw up once at dinner."
))]

updates = interpret(state)
print("Raw fields extracted:")
for k in ("age_months", "temperature_f", "fever_duration_days", "current_medication"):
    if k in updates:
        print(f"  {k}: {updates[k]}")
print()
print("Translated facts:")
for k, v in updates["facts"].items():
    print(f"  {k}: {v}")
print()
print(f"Raw symptoms (for KG grounding): {updates['raw_symptoms']}")
print(f"Missing required facts: {updates['missing_required']}")
print(f"Follow-up question: {updates.get('follow_up_question')}")

Raw fields extracted:
  age_months: 72
  temperature_f: 101.8
  fever_duration_days: 1.0

Translated facts:
  fever: yes
  alert: normal
  intake: normal
  vomiting: once
  breathing: normal

Raw symptoms (for KG grounding): ['fever', 'vomiting']
Missing required facts: ['urination']
Follow-up question: Has your child urinated in the last 8 hours?


## 5. Symbolic rules engine

Alex's 3-layer pipeline lives in `src/rules/rules_agent.py`. It's plain Python — no pyDatalog — for ease of debugging and introspection.

- **Layer 1 (observation):** maps raw facts to predicates like `fever_present`, `poor_intake`, `no_urine`, `lethargy`
- **Layer 2 (concern):** combines observations. Key concern is `dehydration_concern`, which scores `no_urine`/`no_intake` at +2 and `poor_intake`/`repeated_vomiting` at +1. Fever amplifies the score.
- **Layer 3 (decision):** routes concerns to `er_now` | `urgent_eval` | `home_monitor` | `unsupported`

Any `danger_red_flag` (lethargy, breathing difficulty, seizure) short-circuits directly to `er_now`.


In [9]:
from src.rules import rules_agent

# Case A: mild fever, normal everything → home_monitor
case_a = {"facts": {"fever": "yes", "alert": "normal", "breathing": "normal", "intake": "normal", "urination": "normal"}}
out = rules_agent(case_a)
print("Case A — mild fever, no concerns:")
print(f"  observations: {out['observation_predicates']}")
print(f"  concerns:     {out['concern_predicates']}")
print(f"  decision:     {out['decision']}")
print()

# Case B: reduced intake + fever → dehydration amplification → urgent_eval
case_b = {"facts": {"fever": "yes", "alert": "normal", "breathing": "normal", "intake": "reduced", "urination": "normal"}}
out = rules_agent(case_b)
print("Case B — reduced intake + fever (dehydration amplification):")
print(f"  observations: {out['observation_predicates']}")
print(f"  concerns:     {out['concern_predicates']}")
print(f"  decision:     {out['decision']}")
print()

# Case C: lethargy → danger_red_flag → er_now
case_c = {"facts": {"fever": "yes", "alert": "reduced", "breathing": "normal", "intake": "none", "urination": "none"}}
out = rules_agent(case_c)
print("Case C — lethargy + no intake + no urine:")
print(f"  observations: {out['observation_predicates']}")
print(f"  concerns:     {out['concern_predicates']}")
print(f"  decision:     {out['decision']}")

Case A — mild fever, no concerns:
  observations: ['fever_present']
  concerns:     []
  decision:     home_monitor

Case B — reduced intake + fever (dehydration amplification):
  observations: ['fever_present', 'poor_intake']
  concerns:     ['dehydration_concern']
  decision:     urgent_eval

Case C — lethargy + no intake + no urine:
  observations: ['fever_present', 'lethargy', 'no_intake', 'no_urine']
  concerns:     ['danger_red_flag', 'dehydration_concern']
  decision:     er_now


## 6. Safety merge — rules + KG red flags + validation

`evaluate_rules` runs Alex's rules on the facts, then merges with any KG or fallback red flags. Because David's Neo4j synonym matcher uses substring containment, grounding `"fever"` can pull in adjacent concepts like `"febrile seizure"`, spuriously firing red flags like RF_001 (infant fever) or RF_002 (febrile seizure).

To prevent these false positives while still honoring legitimate KG signals, each KG-sourced flag must pass a per-`rule_id` predicate against the actual facts. **Fallback-sourced flags are trusted as-is** since they already checked the facts before being emitted. **Unknown rule_ids fail safe** (rejected) so that schema drift can't silently escalate.


In [10]:
from caretrace.agents.safety import evaluate_rules, _RED_FLAG_VALIDATORS, _validate_red_flag

# Simulate a state where KG spuriously returned RF_001 (infant fever) for a 6yo
state = initial_state()
state["age_months"] = 72  # 6 years old
state["facts"] = {"fever": "yes", "alert": "normal", "breathing": "normal",
                  "intake": "normal", "urination": "normal"}

# Spurious KG red flags (as would happen with substring-matching fever → febrile seizure)
state["kg_red_flags"] = [
    {"rule_id": "RF_001", "description": "Infant fever", "disposition": "er_now", "source": "neo4j"},
    {"rule_id": "RF_002", "description": "Febrile seizure", "disposition": "er_now", "source": "neo4j"},
]

print("Raw KG red flags (before validation):")
for f in state["kg_red_flags"]:
    print(f"  {f['rule_id']} → {f['disposition']}")

print("\nValidation results:")
for f in state["kg_red_flags"]:
    ok = _validate_red_flag(f, state)
    print(f"  {f['rule_id']}: {'APPLY' if ok else 'FILTERED (predicate failed)'}")

print("\nFinal disposition after evaluate_rules:")
out = evaluate_rules(state)
print(f"  disposition: {out['disposition']}")
print(f"  rules_triggered: {out['rules_triggered']}")

Raw KG red flags (before validation):
  RF_001 → er_now
  RF_002 → er_now

Validation results:
  RF_001: FILTERED (predicate failed)
  RF_002: FILTERED (predicate failed)

Final disposition after evaluate_rules:
  disposition: home_monitor
  rules_triggered: ['obs:fever_present', 'safe:no_red_flags', 'decision:home_monitor']


## 7. Explanation agent — structured decision → caregiver text

The explanation agent never makes clinical decisions. It receives a fully-formed decision object from the safety layer (disposition + rule trace + positives + negatives) and asks the LLM to verbalize it in empathetic, plain language the caregiver can act on.


In [11]:
from caretrace.agents.explanation import explain

# Use the Case C result from Section 5 as input to explanation
state = initial_state()
state["age_months"] = 72
state["temperature_f"] = 103.5
state["facts"] = {"fever": "yes", "alert": "reduced", "intake": "none",
                  "breathing": "normal", "urination": "none"}
state.update(evaluate_rules(state))

# Now ask the explanation agent to verbalize
result = explain(state)

# explain() returns an updates dict including a messages list with the AIMessage
for m in result.get("messages", []):
    if hasattr(m, "content"):
        print(m.content)

I'm so sorry to hear that your child is not feeling well. You told me that they have a fever, are less alert than usual, haven't had anything to drink, and haven't urinated in a while. Based on the symptoms you described, I'm concerned that your child may be dehydrated and needs immediate medical attention.

The combination of fever, lethargy, refusal to drink, and no urination is a serious concern. Although their breathing is normal, which is reassuring, the other symptoms outweigh this. 

GO-NOW THRESHOLDS:
  • Breathing becomes fast, labored, or difficult
  • Child has a seizure (shaking, stiffening, eyes rolling)
  • Repeated vomiting — cannot keep any fluids down
  • No urination for 8+ hours
  • Fever rises above 104°F
  • New rash that does not blanch (turn white) when pressed
  • You feel something is seriously wrong — trust your instincts

Please take your child to the emergency room right away. I know this can be scary, but it's crucial to get them the help they need as soon 

## 8. Scenario 1 — Moderate fever, alert, drinking → `home_monitor`

Full multi-turn conversation through the compiled LangGraph, including the LLM interpretation, KG grounding, rule evaluation, and LLM explanation. Watch how the system asks follow-up questions until it has the required facts, then issues a `home_monitor` disposition.


In [12]:
from caretrace.graph import create_app
from caretrace.state import initial_state
from langchain_core.messages import HumanMessage, AIMessage

def render_turn(user_msg, result):
    print(f"\n\x1b[1m[USER]\x1b[0m  {user_msg}")
    for m in reversed(result.get("messages", [])):
        if isinstance(m, AIMessage):
            print(f"\n\x1b[1m[CARETRACE]\x1b[0m")
            for line in m.content.splitlines():
                print(f"  {line}")
            break
    print(f"\n  disposition: \x1b[93m{result.get('disposition') or 'undecided'}\x1b[0m")
    print(f"  phase:       {result.get('phase')}")
    print(f"  facts:       {result.get('facts')}")
    missing = result.get('missing_required', [])
    if missing:
        print(f"  missing:     {missing}")
    rules = result.get('rules_triggered', [])
    if rules:
        print(f"  rule trace:  {rules}")
    flags = result.get('kg_red_flags', [])
    if flags:
        print(f"  red flags:")
        for f in flags:
            print(f"    [{f.get('source', 'kg')}] {f['rule_id']} → {f['disposition']}")

# Run Scenario 1
app, config = create_app(thread_id="nb-s1")
state = initial_state()

scenario_1 = [
    "Hi, my 6-year-old has had a fever since yesterday and threw up once at dinner. I'm worried.",
    "Temperature is 101.8 right now. He's awake and talking to me normally, breathing fine. "
    "He's drinking water like he usually does, just not eating as much solid food.",
    "He just peed about 3 hours ago, normal amount. He's on amoxicillin for an ear infection.",
]

for msg in scenario_1:
    state["messages"] = [HumanMessage(content=msg)]
    result = app.invoke(state, config)
    state.update(result)
    render_turn(msg, result)
    if result.get("is_complete"):
        break

print(f"\n\x1b[1m\x1b[92mFINAL: {state.get('disposition')}\x1b[0m")


[USER]  Hi, my 6-year-old has had a fever since yesterday and threw up once at dinner. I'm worried.

[CARETRACE]
  I'm so glad you're here with us. I just want to check in on a few more things to make sure your little one is comfortable and safe. Can you tell me, how is your child doing right now? Are they awake and able to respond to you when you talk to them, and are they breathing easily, or are you noticing any trouble breathing or rapid breathing?

  disposition: undecided
  phase:       intake
  facts:       {'fever': 'yes', 'vomiting': 'once'}
  missing:     ['alert', 'breathing', 'intake', 'urination']
  rule trace:  ['obs:fever_present', 'decision:unsupported']



[USER]  Temperature is 101.8 right now. He's awake and talking to me normally, breathing fine. He's drinking water like he usually does, just not eating as much solid food.

[CARETRACE]
  I'm so glad to hear that your little one is still alert and breathing normally, and that they're able to drink okay. How's their overall comfort level been, and have they been able to use the bathroom normally? Specifically, do you think they've urinated at all in the last 8 hours?

  disposition: undecided
  phase:       intake
  facts:       {'fever': 'yes', 'vomiting': 'once', 'alert': 'normal', 'intake': 'normal', 'breathing': 'normal'}
  missing:     ['urination']
  rule trace:  ['obs:fever_present', 'decision:unsupported']



[USER]  He just peed about 3 hours ago, normal amount. He's on amoxicillin for an ear infection.

[CARETRACE]
  It's safe to manage your child's symptoms at home with monitoring. You told me that your child has a fever, vomited once, but is still alert, drinking, and urinating normally. Based on the symptoms described, the key concern is the fever, but reassuringly, your child is awake and responding normally, breathing normally, and able to drink fluids.
  
  GO-NOW THRESHOLDS:
    • Child becomes hard to wake or stops responding
    • Breathing becomes fast, labored, or difficult
    • Child has a seizure (shaking, stiffening, eyes rolling)
    • Repeated vomiting — cannot keep any fluids down
    • No urination for 8+ hours
    • Fever rises above 104°F
    • New rash that does not blanch (turn white) when pressed
    • You feel something is seriously wrong — trust your instincts
  
  For overnight care, please follow these steps:
    • Offer small sips of clear fluids (water, elec

## 9. Scenario 2 — High fever, reduced alertness, refusing fluids → `er_now`

Same pipeline, different inputs. The `lethargy` observation triggers `danger_red_flag` directly in the rules layer, yielding `er_now` as soon as the caregiver reports reduced responsiveness.


In [13]:
app, config = create_app(thread_id="nb-s2")
state = initial_state()

scenario_2 = [
    "My 6-year-old is really sick. High fever and she threw up once.",
    "Her temperature is 103.5. She's really out of it — barely responding when I call her. "
    "She won't drink anything. Her breathing seems okay though. "
    "There's a stomach bug going around her school.",
    "I don't think she's peed in about 8 hours.",
]

for msg in scenario_2:
    state["messages"] = [HumanMessage(content=msg)]
    result = app.invoke(state, config)
    state.update(result)
    render_turn(msg, result)
    if result.get("is_complete"):
        break

print(f"\n\x1b[1m\x1b[91mFINAL: {state.get('disposition')}\x1b[0m")


[USER]  My 6-year-old is really sick. High fever and she threw up once.

[CARETRACE]
  I'm so glad you're reaching out for help with your little one. I just want to make sure they're doing okay. Can you tell me a bit more about how they're acting right now? Are they awake and responding normally when you talk to them, and are they having any trouble breathing or seeming to breathe faster than usual?

  disposition: undecided
  phase:       intake
  facts:       {'fever': 'yes', 'vomiting': 'once'}
  missing:     ['alert', 'breathing', 'intake', 'urination']
  rule trace:  ['obs:fever_present', 'decision:unsupported']



[USER]  Her temperature is 103.5. She's really out of it — barely responding when I call her. She won't drink anything. Her breathing seems okay though. There's a stomach bug going around her school.

[CARETRACE]
  I understand you're worried about your child, and I'm here to help. Based on what you've told me, I strongly recommend taking your child to the emergency room right now. You've reported that your child has a fever, has vomited once, is less alert than usual, and hasn't had any intake. These symptoms, especially the fever and reduced alertness, are concerning and need immediate attention.
  
  The system has also identified some key concerns, including the presence of a fever, your child's lethargy, and refusal to take fluids. While it's reassuring that your child's breathing is normal, the other symptoms outweigh this and require prompt medical evaluation.
  
  GO-NOW THRESHOLDS:
    • Breathing becomes fast, labored, or difficult
    • Child has a seizure (shaking, stiffen

## 10. Interpretation agent evaluation — golden dataset sweep

Yoko's golden dataset (`golden_dataset.json`) has 10 `targeted_cases` that stress specific failure modes of the interpreter: implied alertness, temperature unit conversion, never-overwrite semantics, etc. Each case has a `prior_state`, a `caregiver_message`, and an `expected_extracted_fields` delta.

We run the interpretation agent on each case and compute per-field accuracy on the fields the case is asserting.


In [14]:
import json
from caretrace.agents.interpretation import interpret, ExtractionResult
from langchain_core.messages import HumanMessage

with open(REPO_ROOT / "golden_dataset.json") as fh:
    gold = json.load(fh)

# Map Yoko's raw extracted fields (what ExtractionResult produces) onto our
# new interpret() which merges translated facts. For the eval we want the raw
# ExtractionResult, so we call the LLM directly with the same prompt.
import os
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage
from caretrace.agents.interpretation import EXTRACTION_SYSTEM_PROMPT

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, api_key=os.getenv("GROQ_API_KEY"))
extractor = llm.with_structured_output(ExtractionResult)

def extract_from_case(case):
    prior = case["prior_state"]
    # Build a fake history line so the model knows the case context
    history_line = (
        f"Prior known state: " +
        ", ".join(f"{k}={v}" for k, v in prior.items() if v is not None)
    )
    prompt = [
        SystemMessage(content=EXTRACTION_SYSTEM_PROMPT),
        SystemMessage(content=history_line),
        HumanMessage(content=f"Latest caregiver message: {case['caregiver_message']}"),
    ]
    return extractor.invoke(prompt)

# Fields we evaluate (drop never-used ones from the gold schema)
EVAL_FIELDS = ["alert", "temperature_f", "breathing_issues", "urination_8h",
               "drinking", "current_medication", "fever", "vomiting", "rash"]

results = []
for case in gold["targeted_cases"]:
    got = extract_from_case(case)
    exp = case["expected_extracted_fields"]
    row = {"id": case["id"], "category": case["category"]}
    for field in EVAL_FIELDS:
        if field in exp:
            expected = exp[field]
            actual = getattr(got, field, None)
            row[field] = "✓" if actual == expected else f"✗ (got {actual!r}, expected {expected!r})"
    results.append(row)

# Print a summary table
import pandas as pd
df = pd.DataFrame(results).set_index("id")
df

,category,alert,temperature_f,breathing_issues,urination_8h,drinking,current_medication,fever,vomiting,rash
id,,,,,,,,,,
tc_01,implied-field,✓,✓,✓,✓,✓,✓,"✗ (got 'yes', expected None)",✓,✓
tc_02,ambiguous-value,✓,✓,✓,✓,✓,✓,✓,✓,✓
tc_03,non-overwrite-trap,"✗ (got 'yes', expected None)","✗ (got 101.5, expected None)",✓,✓,"✗ (got 'some', expected None)",✓,"✗ (got 'yes', expected None)","✗ (got 'once', expected None)",✓
tc_04,medication-disambiguation,✓,✓,✓,✓,✓,"✗ (got 'none', expected None)",✓,✓,✓
tc_05,hedge-language,✓,✓,✓,"✗ (got 'yes', expected None)",✓,✓,✓,✓,✓
tc_06,multi-field-dump,✓,✓,✓,✓,✓,✓,"✗ (got 'yes', expected None)",✓,✓
tc_07,valid-overwrite,✓,✓,✓,✓,✓,✓,✓,"✗ (got 'repeated', expected None)",✓
tc_08,implied-field,✓,✓,✓,✓,✓,✓,✓,✓,✓
tc_09,vague-numeric,✓,"✗ (got 100.0, expected None)",✓,✓,✓,✓,"✗ (got 'yes', expected None)",✓,✓


In [15]:
# Aggregate accuracy per field
total = len(results)
per_field = {}
for field in EVAL_FIELDS:
    correct = 0
    asserted = 0
    for row in results:
        if field in row:
            asserted += 1
            if row[field] == "✓":
                correct += 1
    if asserted > 0:
        per_field[field] = (correct, asserted, correct / asserted)

print(f"{'field':22s}  {'correct/total':>14s}  {'accuracy':>10s}")
print("-" * 52)
for field, (c, t, acc) in sorted(per_field.items(), key=lambda x: -x[1][2]):
    print(f"{field:22s}  {c:>6d}/{t:<6d}  {acc*100:>8.1f}%")

# Overall
total_correct = sum(c for c, _, _ in per_field.values())
total_asserted = sum(t for _, t, _ in per_field.values())
print("-" * 52)
print(f"{'OVERALL':22s}  {total_correct:>6d}/{total_asserted:<6d}  {total_correct/total_asserted*100:>8.1f}%")

field                    correct/total    accuracy
----------------------------------------------------
breathing_issues            10/10         100.0%
rash                        10/10         100.0%
alert                        9/10          90.0%
urination_8h                 9/10          90.0%
drinking                     9/10          90.0%
current_medication           9/10          90.0%
temperature_f                8/10          80.0%
vomiting                     8/10          80.0%
fever                        6/10          60.0%
----------------------------------------------------
OVERALL                     78/90          86.7%


## 11. Reasoning model comparison — can the LLM replace the rules?

Naveen's suggestion: since the basic LLM already handles *extraction* accurately (87.8%), what happens when we ask a reasoning model to handle the *disposition* step too?

We compare four conditions on the same set of clinical cases:

| Condition | Model | What it sees | Who decides disposition |
|---|---|---|---|
| **A: Standard LLM only** | Llama 3.3 70B | Caregiver transcript + demographics | LLM reasons from scratch |
| **B: Standard LLM + KG** | Llama 3.3 70B | Transcript + grounded SNOMED concepts + red flags | LLM reasons with KG context |
| **C: Neurosymbolic (ours)** | Llama 3.3 70B (extract/verbalize only) | Same as B, but disposition comes from symbolic rules | Rules engine decides |
| **D: Reasoning model + full context** | Qwen3 32B | Transcript + KG + full CPG rule specification | Reasoning model decides with chain-of-thought |

Condition D tests Naveen's specific suggestion: give a reasoning model a one-shot prompt with *all* the context (input, KG findings, and the CPG rules) and see if chain-of-thought reasoning can match the symbolic engine's safety guarantees.

If the reasoning model matches the symbolic rules on all cases including edge cases, that suggests the rules could be "compiled into" a sufficiently capable prompt. If it still diverges, that's evidence for keeping deterministic symbolic reasoning in the safety-critical path.

In [16]:
import os, json
from pydantic import BaseModel, Field
from typing import Optional
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from groq import Groq

# Pydantic schema for structured disposition output (Llama 3.3 — supports tool calling)
class LLMDisposition(BaseModel):
    disposition: str = Field(
        description="One of: 'er_now', 'urgent_eval', or 'home_monitor'"
    )
    reasoning: str = Field(
        description="2-3 sentence clinical reasoning for this disposition"
    )
    confidence: str = Field(
        description="'high', 'medium', or 'low'"
    )

# ── Standard LLM (Llama 3.3 70B) — used for Conditions A and B ──────────────
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0,
               api_key=os.getenv("GROQ_API_KEY"))
reasoner = llm.with_structured_output(LLMDisposition)

# ── Reasoning model (Qwen3 32B) — used for Condition D ──────────────────────
# Qwen3 doesn't support tool calling on Groq, so we use JSON mode + manual parsing.
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
REASONING_MODEL = "qwen/qwen3-32b"

def invoke_reasoning_model(system_prompt: str, user_prompt: str) -> dict:
    """Call the reasoning model with JSON mode and parse the response."""
    response = groq_client.chat.completions.create(
        model=REASONING_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

# ── Shared disposition rules text ────────────────────────────────────────────

DISPOSITION_RULES_TEXT = """\
Available dispositions (choose exactly one):
- er_now: Go to the Emergency Room immediately. Use for: lethargy/unresponsive,
  seizure, breathing difficulty, infant (<3 months) with any fever, severe dehydration
  (no urine AND no intake).
- urgent_eval: See a doctor urgently within a few hours. Use for: dehydration concern
  (reduced intake or no urine, especially with fever), persistent fever >5 days,
  repeated vomiting preventing oral rehydration.
- home_monitor: Safe to manage at home with monitoring. Use when: child is alert,
  breathing normally, drinking fluids, urinating normally, and no red flags.
"""

# ── Condition A prompt: standard LLM, transcript only ────────────────────────

PROMPT_A = """\
You are an expert pediatric triage nurse. Based ONLY on the clinical information
provided, determine the appropriate triage disposition for this child.

""" + DISPOSITION_RULES_TEXT + """
Think step by step about what symptoms are present and what they mean clinically.
"""

# ── Condition B prompt: standard LLM + KG context ───────────────────────────

PROMPT_B = """\
You are an expert pediatric triage nurse with access to a medical knowledge graph.
Based on the clinical information AND the knowledge graph findings below,
determine the appropriate triage disposition.

""" + DISPOSITION_RULES_TEXT + """
The knowledge graph has grounded the symptoms to SNOMED CT concepts and checked
for clinical red flags. Use this information alongside the clinical presentation.
"""

# ── Condition D prompt: reasoning model + full CPG rules ─────────────────────
# This is Naveen's suggestion: one-shot prompt with ALL context (input, KG, rules).
# The reasoning model gets the complete CPG rule specification so it can apply
# the same logic the symbolic engine uses.

PROMPT_D = """\
You are an expert pediatric triage nurse performing evidence-based triage using
the Seattle Children's Fever CPG (Clinical Practice Guideline).

You will receive: (1) the caregiver's report, (2) knowledge graph findings from
SNOMED CT, and (3) the complete triage rule specification. Apply the rules
systematically using chain-of-thought reasoning.

RESPOND WITH JSON ONLY: {"disposition": "er_now|urgent_eval|home_monitor", "reasoning": "...", "confidence": "high|medium|low"}

═══════════════════════════════════════════════════════════════════════════════
TRIAGE RULE SPECIFICATION (from Seattle Children's Fever CPG)
═══════════════════════════════════════════════════════════════════════════════

LAYER 1 — OBSERVATION PREDICATES (extract from the case):
  fever_present       := temperature >= 100.4°F OR caregiver reports fever
  poor_intake         := child drinking reduced OR refusing fluids
  no_urine            := no urination in 8+ hours
  vomiting_repeated   := vomiting more than once
  not_alert           := child lethargic, barely responding, hard to wake
  breathing_difficulty := labored breathing, retractions, fast breathing
  seizure_present     := any seizure activity reported
  rash_present        := new rash reported
  infant              := age < 3 months

LAYER 2 — CONCERN PREDICATES (derived from observations):
  dehydration_concern := poor_intake OR no_urine
  dehydration_severe  := no_urine AND poor_intake (intake = none)
  NOTE: dehydration_concern is amplified when fever_present (fever increases
        fluid losses, making even mild dehydration more clinically significant)

LAYER 3 — DECISION RULES (apply in priority order — first match wins):
  1. infant AND fever_present                    → er_now    (bright-line rule: ANY fever in <3mo)
  2. not_alert                                   → er_now    (altered mental status)
  3. seizure_present                             → er_now
  4. breathing_difficulty                        → er_now
  5. dehydration_severe                          → er_now    (no urine + refusing all fluids)
  6. dehydration_concern AND fever_present       → urgent_eval (dehydration risk amplified by fever)
  7. dehydration_concern (without fever)         → urgent_eval
  8. vomiting_repeated AND poor_intake           → urgent_eval
  9. all observations normal, no red flags       → home_monitor

CRITICAL: Rule 1 is a BRIGHT-LINE rule — any infant (<3 months) with ANY fever
goes to the ER regardless of how well they appear. This is non-negotiable in
pediatric practice because neonatal infections can deteriorate rapidly.
═══════════════════════════════════════════════════════════════════════════════
"""

# ── Test cases ───────────────────────────────────────────────────────────────

TEST_CASES = [
    {
        "id": "S1",
        "label": "Moderate fever, alert, drinking normally",
        "transcript": "My 6-year-old has had a fever since yesterday (101.8°F). "
                      "He threw up once. He's awake and talking to me normally, "
                      "breathing fine, drinking water normally. He peed 3 hours ago. "
                      "He's on amoxicillin for an ear infection.",
        "age_months": 72,
        "facts": {"fever": "yes", "alert": "normal", "breathing": "normal",
                  "intake": "normal", "urination": "normal", "vomiting": "once"},
        "expected": "home_monitor",
    },
    {
        "id": "S1b",
        "label": "Reduced intake + fever (dehydration amplification)",
        "transcript": "My 6-year-old has a 101.8°F fever. She's awake and talking, "
                      "breathing fine, but she's only taking small sips of water. "
                      "She's urinating normally.",
        "age_months": 72,
        "facts": {"fever": "yes", "alert": "normal", "breathing": "normal",
                  "intake": "reduced", "urination": "normal"},
        "expected": "urgent_eval",
    },
    {
        "id": "S2",
        "label": "Lethargy + no intake → danger",
        "transcript": "My 6-year-old has a 103.5°F fever. She's really out of it — "
                      "barely responding when I call her name. She won't drink anything. "
                      "Breathing seems okay. She hasn't peed in 8 hours.",
        "age_months": 72,
        "facts": {"fever": "yes", "alert": "reduced", "intake": "none",
                  "breathing": "normal", "urination": "none", "vomiting": "once"},
        "expected": "er_now",
    },
    {
        "id": "S3",
        "label": "Seizure (red flag short-circuit)",
        "transcript": "My 2-year-old had a fever and then started shaking all over "
                      "for about 30 seconds. He seems drowsy now.",
        "age_months": 24,
        "facts": {"fever": "yes", "seizure": "yes"},
        "expected": "er_now",
    },
    {
        "id": "S4",
        "label": "Infant (<3mo) with any fever",
        "transcript": "My 2-month-old has a temperature of 101°F. She's eating less "
                      "than usual but still taking some milk.",
        "age_months": 2,
        "facts": {"fever": "yes"},
        "expected": "er_now",
    },
    {
        "id": "S5",
        "label": "Breathing difficulty",
        "transcript": "My 4-year-old has a 102°F fever and is working really hard to "
                      "breathe — I can see his ribs pulling in. He's alert though.",
        "age_months": 48,
        "facts": {"fever": "yes", "breathing": "difficulty", "alert": "normal"},
        "expected": "er_now",
    },
]

print(f"Defined {len(TEST_CASES)} test cases for comparison")
print(f"Standard LLM:    Llama 3.3 70B (via Groq)")
print(f"Reasoning model: Qwen3 32B (via Groq)")

Defined 6 test cases for comparison
Standard LLM:    Llama 3.3 70B (via Groq)
Reasoning model: Qwen3 32B (via Groq)


In [17]:
from caretrace.agents.safety import evaluate_rules
from caretrace.agents.knowledge import normalize
from caretrace.state import initial_state
from caretrace.kg.fallback_red_flags import check_fallback_red_flags

import os
os.environ["CARETRACE_KG_BACKEND"] = "dict"  # deterministic for comparison
# Reset the module-level adapter so it picks up the env var
from caretrace.agents import knowledge as _know_mod
_know_mod._adapter = None

rows = []

for case in TEST_CASES:
    print(f"\n{'='*70}")
    print(f"  {case['id']}: {case['label']}")
    print(f"  Expected: {case['expected']}")
    print(f"{'='*70}")

    row = {"id": case["id"], "label": case["label"], "expected": case["expected"]}

    # ── Condition A: LLM-only (transcript + demographics) ────────────────
    prompt_a = [
        SystemMessage(content=PROMPT_A),
        HumanMessage(content=(
            f"Patient: {case['age_months']} months old\n\n"
            f"Caregiver says: \"{case['transcript']}\""
        )),
    ]
    try:
        res_a = reasoner.invoke(prompt_a)
        row["llm_only"] = res_a.disposition
        row["llm_only_reasoning"] = res_a.reasoning
        row["llm_only_confidence"] = res_a.confidence
    except Exception as e:
        row["llm_only"] = f"ERROR: {e}"
        row["llm_only_reasoning"] = ""
        row["llm_only_confidence"] = ""

    print(f"  A (LLM-only):    {row['llm_only']:15s}  [{row.get('llm_only_confidence', '')}]")

    # ── Condition B: LLM + KG context ────────────────────────────────────
    # Build KG context from the case facts
    state_b = initial_state()
    state_b["age_months"] = case["age_months"]
    state_b["facts"] = dict(case["facts"])
    # Generate raw_symptoms deterministically
    from caretrace.agents.interpretation import _derive_raw_symptoms
    state_b["raw_symptoms"] = _derive_raw_symptoms(case["facts"], case.get("temperature_f"))
    state_b.update(normalize(state_b))

    kg_context = ""
    grounded = state_b.get("grounded_concepts", [])
    if grounded:
        kg_context += "Grounded SNOMED concepts:\n"
        for g in grounded:
            kg_context += f"  - {g.get('mention', '?')} → {[c.get('fsn', '?') for c in g.get('concepts', [])]}\n"

    flags = state_b.get("kg_red_flags", [])
    if flags:
        kg_context += "\nKG Red flags triggered:\n"
        for f in flags:
            kg_context += f"  - [{f.get('source')}] {f['rule_id']}: {f.get('description', '')} → {f['disposition']}\n"

    prompt_b = [
        SystemMessage(content=PROMPT_B),
        HumanMessage(content=(
            f"Patient: {case['age_months']} months old\n\n"
            f"Caregiver says: \"{case['transcript']}\"\n\n"
            f"Knowledge graph findings:\n{kg_context}"
        )),
    ]
    try:
        res_b = reasoner.invoke(prompt_b)
        row["llm_kg"] = res_b.disposition
        row["llm_kg_reasoning"] = res_b.reasoning
        row["llm_kg_confidence"] = res_b.confidence
    except Exception as e:
        row["llm_kg"] = f"ERROR: {e}"
        row["llm_kg_reasoning"] = ""
        row["llm_kg_confidence"] = ""

    print(f"  B (LLM+KG):      {row['llm_kg']:15s}  [{row.get('llm_kg_confidence', '')}]")

    # ── Condition C: Symbolic pipeline (ground truth) ────────────────────
    state_c = initial_state()
    state_c["age_months"] = case["age_months"]
    state_c["facts"] = dict(case["facts"])
    state_c["raw_symptoms"] = _derive_raw_symptoms(case["facts"], case.get("temperature_f"))
    state_c.update(normalize(state_c))
    state_c.update(evaluate_rules(state_c))

    symbolic = state_c.get("disposition") or "unsupported"
    row["symbolic"] = symbolic
    row["rules_trace"] = state_c.get("rules_triggered", [])

    print(f"  C (Symbolic):     {symbolic:15s}  trace: {row['rules_trace']}")

    # ── Condition D: Reasoning model + full context (Naveen's suggestion) ─
    # One-shot prompt with transcript + KG findings + full CPG rule spec
    user_prompt_d = (
        f"Patient: {case['age_months']} months old\n\n"
        f"Caregiver says: \"{case['transcript']}\"\n\n"
    )
    if kg_context:
        user_prompt_d += f"Knowledge graph findings:\n{kg_context}\n\n"
    user_prompt_d += "Apply the triage rules systematically. Which rule fires first?"

    try:
        res_d = invoke_reasoning_model(PROMPT_D, user_prompt_d)
        row["reasoning_model"] = res_d.get("disposition", "ERROR")
        row["reasoning_model_reasoning"] = res_d.get("reasoning", "")
        row["reasoning_model_confidence"] = res_d.get("confidence", "")
    except Exception as e:
        row["reasoning_model"] = f"ERROR: {e}"
        row["reasoning_model_reasoning"] = ""
        row["reasoning_model_confidence"] = ""

    print(f"  D (Reasoning):    {row['reasoning_model']:15s}  [{row.get('reasoning_model_confidence', '')}]")

    # ── Match check ──────────────────────────────────────────────────────
    row["a_correct"] = row["llm_only"] == case["expected"]
    row["b_correct"] = row["llm_kg"] == case["expected"]
    row["c_correct"] = symbolic == case["expected"]
    row["d_correct"] = row["reasoning_model"] == case["expected"]

    rows.append(row)

# Restore auto backend
del os.environ["CARETRACE_KG_BACKEND"]
_know_mod._adapter = None


  S1: Moderate fever, alert, drinking normally
  Expected: home_monitor


  A (LLM-only):    home_monitor     [high]


  B (LLM+KG):      home_monitor     [high]
  C (Symbolic):     home_monitor     trace: ['obs:fever_present', 'safe:no_red_flags', 'decision:home_monitor']


  D (Reasoning):    home_monitor     [high]

  S1b: Reduced intake + fever (dehydration amplification)
  Expected: urgent_eval


  A (LLM-only):    urgent_eval      [medium]


...

  B (LLM+KG):      er_now           [high]
  C (Symbolic):     er_now           trace: ['obs:fever_present', 'decision:unsupported', 'fallback:FB_INFANT_FEVER→er_now']


  D (Reasoning):    er_now           [high]

  S5: Breathing difficulty
  Expected: er_now


  A (LLM-only):    er_now           [high]


  B (LLM+KG):      er_now           [high]
  C (Symbolic):     er_now           trace: ['obs:fever_present', 'decision:unsupported', 'fallback:FB_BREATHING→er_now']


  D (Reasoning):    er_now           [high]


In [18]:
import pandas as pd

# ── Summary table ────────────────────────────────────────────────────────────
summary = []
for r in rows:
    summary.append({
        "Case": f"{r['id']}: {r['label'][:35]}",
        "Expected": r["expected"],
        "A: LLM-only": ("✓ " if r["a_correct"] else "✗ ") + r.get("llm_only", "?"),
        "B: LLM+KG":   ("✓ " if r["b_correct"] else "✗ ") + r.get("llm_kg", "?"),
        "C: Symbolic":  ("✓ " if r["c_correct"] else "✗ ") + r.get("symbolic", "?"),
        "D: Reasoning": ("✓ " if r["d_correct"] else "✗ ") + r.get("reasoning_model", "?"),
    })

df_cmp = pd.DataFrame(summary)
print(df_cmp.to_string(index=False))

# ── Accuracy summary ────────────────────────────────────────────────────────
n = len(rows)
a_acc = sum(r["a_correct"] for r in rows) / n * 100
b_acc = sum(r["b_correct"] for r in rows) / n * 100
c_acc = sum(r["c_correct"] for r in rows) / n * 100
d_acc = sum(r["d_correct"] for r in rows) / n * 100

print(f"\n{'Condition':30s}  {'Model':25s}  {'Accuracy':>10s}")
print("-" * 70)
print(f"{'A: Standard LLM only':30s}  {'Llama 3.3 70B':25s}  {a_acc:>8.1f}%")
print(f"{'B: Standard LLM + KG':30s}  {'Llama 3.3 70B':25s}  {b_acc:>8.1f}%")
print(f"{'C: Neurosymbolic (ours)':30s}  {'Symbolic rules':25s}  {c_acc:>8.1f}%")
print(f"{'D: Reasoning model + rules':30s}  {'Qwen3 32B':25s}  {d_acc:>8.1f}%")

                                    Case     Expected    A: LLM-only      B: LLM+KG    C: Symbolic   D: Reasoning
 S1: Moderate fever, alert, drinking nor home_monitor ✓ home_monitor ✓ home_monitor ✓ home_monitor ✓ home_monitor
S1b: Reduced intake + fever (dehydration  urgent_eval  ✓ urgent_eval  ✓ urgent_eval  ✓ urgent_eval  ✓ urgent_eval
       S2: Lethargy + no intake → danger       er_now       ✓ er_now       ✓ er_now       ✓ er_now       ✓ er_now
    S3: Seizure (red flag short-circuit)       er_now       ✓ er_now       ✓ er_now       ✓ er_now       ✓ er_now
        S4: Infant (<3mo) with any fever       er_now  ✗ urgent_eval       ✓ er_now       ✓ er_now       ✓ er_now
                S5: Breathing difficulty       er_now       ✓ er_now       ✓ er_now       ✓ er_now       ✓ er_now

Condition                       Model                        Accuracy
----------------------------------------------------------------------
A: Standard LLM only            Llama 3.3 70B               

In [19]:
# ── Show reasoning model chain-of-thought on key cases ─────────────────────
print("REASONING MODEL (Qwen3 32B) — chain-of-thought on each case:\n")
for r in rows:
    match = "✓" if r["d_correct"] else "✗"
    print(f"  {match} {r['id']}: {r['label']}")
    print(f"    Disposition: {r.get('reasoning_model', '?')} (expected: {r['expected']})")
    print(f"    Reasoning:   \"{r.get('reasoning_model_reasoning', '')}\"")
    print()

# ── Divergence analysis ────────────────────────────────────────────────────
divergent_a = [r for r in rows if not r["a_correct"]]
divergent_d = [r for r in rows if not r["d_correct"]]

print("=" * 70)
if divergent_a:
    print(f"\nSTANDARD LLM (Llama 3.3) MISSES ({len(divergent_a)} case(s)):")
    for r in divergent_a:
        print(f"  {r['id']}: predicted {r['llm_only']}, expected {r['expected']}")
        print(f"    Reasoning: \"{r.get('llm_only_reasoning', '')}\"")
else:
    print("\nStandard LLM matched symbolic pipeline on all cases.")

if divergent_d:
    print(f"\nREASONING MODEL (Qwen3) MISSES ({len(divergent_d)} case(s)):")
    for r in divergent_d:
        print(f"  {r['id']}: predicted {r['reasoning_model']}, expected {r['expected']}")
        print(f"    Reasoning: \"{r.get('reasoning_model_reasoning', '')}\"")
else:
    print("\nReasoning model matched symbolic pipeline on all cases.")

print("\n" + "=" * 70)
print("KEY INSIGHT:")
if not divergent_d and divergent_a:
    print("  The reasoning model (Qwen3 32B) with full CPG rules matches the")
    print("  symbolic pipeline perfectly, while the standard LLM (Llama 3.3 70B)")
    print("  misses edge cases. This suggests a sufficiently capable reasoning")
    print("  model CAN internalize clinical rules from a prompt — but the symbolic")
    print("  approach still wins on: determinism, auditability, and zero risk of")
    print("  prompt drift across model updates.")
elif not divergent_d and not divergent_a:
    print("  Both models matched on all 6 cases. The test set may need harder")
    print("  edge cases to differentiate LLM reasoning from symbolic rules.")
else:
    print("  Even the reasoning model with full rules specification diverges from")
    print("  the symbolic pipeline — evidence that encoding rules in code provides")
    print("  stronger safety guarantees than prompt engineering alone.")

REASONING MODEL (Qwen3 32B) — chain-of-thought on each case:

  ✓ S1: Moderate fever, alert, drinking normally
    Disposition: home_monitor (expected: home_monitor)
    Reasoning:   "The patient is a 6-year-old with a fever (101.8°F) and one episode of vomiting. He is alert, breathing normally, has normal fluid intake, and urinated 3 hours ago. No dehydration concerns (poor_intake=false, no_urine=false). Vomiting_repeated is false (only one episode). No other red flags (not an infant, no seizures, no breathing difficulty, no severe dehydration). Rule 9 applies as all observations are normal with no urgent concerns."

  ✓ S1b: Reduced intake + fever (dehydration amplification)
    Disposition: urgent_eval (expected: urgent_eval)
    Reasoning:   "The child has a fever (101.8°F) and reduced fluid intake (small sips of water), meeting dehydration_concern. Per rule 6, dehydration_concern combined with fever_present requires urgent evaluation. The child is not an infant (<3mo), has no seve

## 12. Limitations and future work

**Current limitations**

- **KG substring matching.** David's Neo4j synonym index uses substring containment, so `"fever"` matches both `Fever` and `Febrile convulsion`. The safety layer now gates every KG-sourced red flag through per-`rule_id` predicates against the actual facts, but the long-term fix is to switch to token/phrase matching at the KG layer.
- **Alex's conservative dehydration scoring.** `poor_intake + fever_present` alone is enough to trigger `dehydration_concern → urgent_eval`, even with normal urine output and no vomiting. This is clinically reasonable but strict; many "borderline" home cases land in `urgent_eval`.
- **Interpretation agent never overwrites** — missing fields stay missing across turns. This means a caregiver can't say "actually, he's drinking normally now" and have the system update a previous `intake=reduced`. A future turn-awareness layer should reconcile explicit corrections.
- **Explanation hallucination risk.** The LLM verbalizer is given a strict JSON decision object, but it still generates free text. We rely on the system prompt ("only reword the structured decision") to prevent it from adding clinical judgments. A template-based fallback would be safer.
- **Dosing.** The KG has acetaminophen/ibuprofen dosing concepts but the pipeline doesn't yet surface weight-based dose recommendations. The `weight_kg` field is wired but unused.

**Future work**

1. **Rule + red-flag unification** into a single declarative format (YAML or JSON) so Alex's rules, David's KG rules, and the fallback rules can all be validated against the same test matrix.
2. **Turn-aware interpretation** that distinguishes "new information" from "correction" so the agent can properly update prior facts.
3. **Counterfactual explanations.** "What would need to change for this to be a `home_monitor` case?" — directly leverages the symbolic trace.
4. **End-to-end evaluation.** The current golden dataset covers the interpretation layer only. We need disposition-level golden cases that exercise the whole pipeline.
5. **Safe deployment.** This notebook is a research prototype, not a medical device. For any real deployment we'd need human-in-the-loop oversight, clinician validation of every rule, and clear scoping to "decision support" not "decision-making".

---

**Repo:** `neurosym-finalproject/` · **Package:** `caretrace/` · **Run the interactive CLI:** `python -m caretrace.app`